# Estimando distribuciones (parte 1)
El objetivo de esta notebook es explorar una primera manera de aproximar $p(y|x)$ y $p(x|y)$ en un set de datos tabular. En este set de datos $x$ tiene valores discretos, $x\in\mathbb{D}^k$, y el target $y$ es un booleano, $y\in\{0,1\}$.

## Imports

In [1]:
import numpy as np
import pandas as pd
import random

## Cargamos los datos

In [2]:
df = pd.read_csv('./tennis.csv', delimiter=',', header=0)
df

,Day,Outlook,Temp,Humidity,Wind,Tennis
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes
5,D6,Rain,Cool,Normal,Strong,No
6,D7,Overcast,Cool,Normal,Strong,Yes
7,D8,Sunny,Mild,High,Weak,No
8,D9,Sunny,Cool,Normal,Weak,Yes
9,D10,Rain,Mild,Normal,Weak,Yes


### Eliminamos la columna Day 

In [3]:
df = df.drop('Day', axis=1)
df

,Outlook,Temp,Humidity,Wind,Tennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [4]:
X_names = df.columns.to_list()[:-1]
X_names

['Outlook', 'Temp', 'Humidity', 'Wind']

Guardamos en la variable $X$ todas las features del dataset.

In [5]:
X = df.iloc[:,0:-1]
X

,Outlook,Temp,Humidity,Wind
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rain,Mild,High,Weak
4,Rain,Cool,Normal,Weak
5,Rain,Cool,Normal,Strong
6,Overcast,Cool,Normal,Strong
7,Sunny,Mild,High,Weak
8,Sunny,Cool,Normal,Weak
9,Rain,Mild,Normal,Weak


In [6]:
Y_name = df.columns.to_list()[-1]
Y_name

'Tennis'

Guardamos en $Y$ el objetivo

In [7]:
Y = df.iloc[:,-1]
Y

0      No
1      No
2     Yes
3     Yes
4     Yes
5      No
6     Yes
7      No
8     Yes
9     Yes
10    Yes
11    Yes
12    Yes
13     No
Name: Tennis, dtype: object

## Construimos una tabla de observaciones

En este paso vamos a crear una tabla de observaciones. Esta tabla tiene que contener la frecuencia de cada observación. Para este ejemplo tomaremos a $x$ como **Outlook**.

Calcule las dimensiones de la tabla

In [8]:
# Cantidad total de elementos
N = len(df)

# Elementos únicos de la clase Outlook
xvalues = df['Outlook'].unique()
dimx = len(xvalues)

# Elementos únicos del objetivo
yvalues = df['Tennis'].unique()
dimy = len(yvalues)

print(f'Cantidad total de elementos: {N}')
print(f'Elementos únicos de la clase Outlook: {xvalues}')
print(f'Elementos únicos del objetivo: {yvalues}')

Cantidad total de elementos: 14
Elementos únicos de la clase Outlook: ['Sunny' 'Overcast' 'Rain']
Elementos únicos del objetivo: ['No' 'Yes']


Calculamos la tabla de frecuencia.

In [9]:
obs = pd.DataFrame(0, columns=yvalues, index=xvalues)

for _, row in df.iterrows():
    obs.loc[row['Outlook'], row['Tennis']] += 1

obs

,No,Yes
Sunny,3,2
Overcast,0,4
Rain,2,3


## Aproximación de la distribución conjunta $p(x,y)$

Tome a $x$ como Outlook y aproxime la distribución conjunta utilizando la tabla de observaciones. 

In [10]:
joint_x_y = obs / N
joint_x_y

,No,Yes
Sunny,0.214286,0.142857
Overcast,0.000000,0.285714
Rain,0.142857,0.214286


1. ¿Qué significa el valor calculado en los índices "Sunny", "Yes"?

2. ¿Justifique el resultado de la pareja "Overcast", "No"?

**Respuestas:**

1. Ese valor (0.142857 o sea 2/14) es la probabilidad conjunta de que esté Sunny Y se juegue al tenis. Básicamente significa que de todas las observaciones, un 14.3% son días soleados donde se jugó.

2. Da cero porque en nuestros datos nunca pasó que estuviera Overcast y NO se jugara al tenis. Todos los días Overcast se jugó, entonces la probabilidad conjunta es 0.

## Aproximamos $p(y|x)$

Tome a $x$ como **Outlook** y estime la probabilidad condicional de $y$ dado $x$. Luego realice una muestra de 10 valores de $y$ dado $x = Sunny$.

Calculamos la cantidad de entradas por cada valor distinto de $x$.

In [11]:
m = obs.sum(axis=1)
obs['m'] = m
m

Sunny       5
Overcast    4
Rain        5
dtype: int64

Calculamos la cantidad de entradas por cada valor distinto de $y$.

In [12]:
l = obs.sum(axis=0)
obs.loc['l'] = l
l

No      5
Yes     9
m      14
dtype: int64

In [13]:
obs

,No,Yes,m
Sunny,3,2,5
Overcast,0,4,4
Rain,2,3,5
l,5,9,14


Calcule la probabilidad condicional de $y$ dado $x$.

In [14]:
p_y_x = pd.DataFrame(0, columns=yvalues, index=xvalues)

for x in xvalues:
    total = obs.loc[x, 'm']
    for y in yvalues:
        p_y_x.loc[x, y] = obs.loc[x, y] / total

p_y_x

C:\Users\Enrique\AppData\Local\Temp\ipykernel_77020\616825711.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.6' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  p_y_x.loc[x, y] = obs.loc[x, y] / total
C:\Users\Enrique\AppData\Local\Temp\ipykernel_77020\616825711.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.4' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  p_y_x.loc[x, y] = obs.loc[x, y] / total


,No,Yes
Sunny,0.6,0.4
Overcast,0.0,1.0
Rain,0.4,0.6


3. ¿La suma de cada fila siempre tiene que dar 1? ¿Por qué?

4. ¿Y si la suma de las columnas?

**Respuestas:**

3. Sí, siempre tiene que dar 1. Es porque P(y|x) significa que dado un x específico, alguno de los valores de y tiene que pasar. Si está Sunny, o se juega o no se juega, no hay otra opción. Entonces sumando todas las probabilidades de y para un x fijo, tiene que dar 100% (o sea 1).

4. Las columnas no tienen por qué sumar 1. Acá estamos condicionando por las filas (por x), no por las columnas. Cada fila es una distribución de probabilidad independiente sobre y.

Realice 10 muestras de $y$ dado $x = Sunny$.

Puede utilizar la función random.choice de numpy. https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html

In [15]:
sampled_values = np.random.choice(yvalues, size=10, p=p_y_x.loc['Sunny'].values)

sampled_values

array(['No', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes'],
      dtype=object)

5. ¿Qué pasaría si utilizamos $x = Overcast$ en vez de $x = Sunny$? ¿Tiene sentido que pase esto? ¿Por qué?

**Respuesta:**

Si muestreamos con x=Overcast, siempre va a salir "Yes". Eso pasa porque P(Yes|Overcast) = 1.0 y P(No|Overcast) = 0.0. Tiene total sentido, porque en los datos que tenemos, todos los días Overcast se jugó al tenis, nunca hubo un día nublado donde no se jugara. Entonces cuando muestreamos, estamos replicando ese patrón que vimos en las observaciones.

## Aproximamos $p(x|y)$
Tome a $x$ como Outlook y estime la probabilidad condicional de $x$ dado $y$ basandose en la tabla de observaciones. Luego realice 10 muestras de $x$ dado $y = Yes$.

$p(x|y)$

In [16]:
p_x_y = pd.DataFrame(0, columns=yvalues, index=xvalues)

for y in yvalues:
    total = obs.loc['l', y]
    for x in xvalues:
        p_x_y.loc[x, y] = obs.loc[x, y] / total

p_x_y

C:\Users\Enrique\AppData\Local\Temp\ipykernel_77020\3866335197.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.6' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  p_x_y.loc[x, y] = obs.loc[x, y] / total
C:\Users\Enrique\AppData\Local\Temp\ipykernel_77020\3866335197.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2222222222222222' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  p_x_y.loc[x, y] = obs.loc[x, y] / total


,No,Yes
Sunny,0.6,0.222222
Overcast,0.0,0.444444
Rain,0.4,0.333333


Muestreo

In [17]:
sampled_values = np.random.choice(xvalues, size=10, p=p_x_y['Yes'].values)

sampled_values

array(['Rain', 'Rain', 'Overcast', 'Overcast', 'Overcast', 'Sunny',
       'Rain', 'Rain', 'Overcast', 'Rain'], dtype=object)

## Aproxime $p(y,o,h,w,t)$

Aproxime la proabilidad conjunta del Tennis (y), Outlook (o), Humidity (h), Wind (w), Temp (t).

Recuerde que $p(y,o,h,w,t)$ = $p(y)$.$p(o|y)$.$p(h|y,o)$.$p(w|y,o,h)$.$p(t|y,o,h,w)$

Calcule P(y)

In [18]:
# P(Y)
p_y = df['Tennis'].value_counts() / N
p_y

Tennis
Yes    0.642857
No     0.357143
Name: count, dtype: float64

Calcule P(o|y)

In [19]:
p_o_y = pd.crosstab(df['Outlook'], df['Tennis'], normalize='columns')
p_o_y

Tennis,No,Yes
Outlook,,
Overcast,0.0,0.444444
Rain,0.4,0.333333
Sunny,0.6,0.222222


Calcule P(h|y,o)

Recomendamos usar la función *crosstab* de pandas. En este link pueden encontrar un ejemplo de su uso: https://www.geeksforgeeks.org/pandas-crosstab-function-in-python/

In [20]:
# Calcule la tabla de frecuencia
p_h_yo = pd.crosstab([df['Tennis'], df['Outlook']], df['Humidity'])

# Luego se divide cada fila por la suma de sus elementos, puede usar las funciones div y sum de pandas
p_h_yo = p_h_yo.div(p_h_yo.sum(axis=1), axis=0)

# No se oliden de llenar los valores NaN!!!
p_h_yo = p_h_yo.fillna(0)

# Cambiamos los nombres de los indices (si es que lo precisa)
p_h_yo.index.names = ['Tennis', 'Outlook']

p_h_yo

Humidity             High    Normal
Tennis Outlook                     
No     Rain      0.500000  0.500000
       Sunny     1.000000  0.000000
Yes    Overcast  0.500000  0.500000
       Rain      0.333333  0.666667
       Sunny     0.000000  1.000000

Calcule P(w|y,o,h)

In [21]:
p_w_yoh = pd.crosstab([df['Tennis'], df['Outlook'], df['Humidity']], df['Wind'])
p_w_yoh = p_w_yoh.div(p_w_yoh.sum(axis=1), axis=0).fillna(0)
p_w_yoh

Wind                        Strong      Weak
Tennis Outlook  Humidity                    
No     Rain     High      1.000000  0.000000
                Normal    1.000000  0.000000
       Sunny    High      0.333333  0.666667
Yes    Overcast High      0.500000  0.500000
                Normal    0.500000  0.500000
       Rain     High      0.000000  1.000000
                Normal    0.000000  1.000000
       Sunny    Normal    0.500000  0.500000

Calcule P(t|y,o,h,w)

In [22]:
p_t_yohw = pd.crosstab([df['Tennis'], df['Outlook'], df['Humidity'], df['Wind']], df['Temp'])
p_t_yohw = p_t_yohw.div(p_t_yohw.sum(axis=1), axis=0).fillna(0)
p_t_yohw

Temp                             Cool  Hot  Mild
Tennis Outlook  Humidity Wind                   
No     Rain     High     Strong   0.0  0.0   1.0
                Normal   Strong   1.0  0.0   0.0
       Sunny    High     Strong   0.0  1.0   0.0
                         Weak     0.0  0.5   0.5
Yes    Overcast High     Strong   0.0  0.0   1.0
                         Weak     0.0  1.0   0.0
                Normal   Strong   1.0  0.0   0.0
                         Weak     0.0  1.0   0.0
       Rain     High     Weak     0.0  0.0   1.0
                Normal   Weak     0.5  0.0   0.5
       Sunny    Normal   Strong   0.0  0.0   1.0
                         Weak     1.0  0.0   0.0

Calcule P(y,o,h,w,t)

In [23]:
# Definimos una función que nos calcula la probabilidad conjunta usando la regla del producto.
def calculate_prob(y,o,h,w,t):
    "Calculate the probability of occurrence of a row"
    prob_y = p_y[y]
    prob_o_y = p_o_y.loc[o, y]
    prob_h_yo = p_h_yo.loc[(y, o), h]
    prob_w_yoh = p_w_yoh.loc[(y, o, h), w]
    prob_t_yohw = p_t_yohw.loc[(y, o, h, w), t]
    
    return prob_y * prob_o_y * prob_h_yo * prob_w_yoh * prob_t_yohw

In [24]:
prob = calculate_prob('Yes','Sunny','Normal','Weak','Cool')

print(f'P(Yes|Sunny,Normal,Weak,Cool) = {prob}')
print(f"Cantidad de observaciones: {prob*N}")

P(Yes|Sunny,Normal,Weak,Cool) = 0.07142857142857142
Cantidad de observaciones: 1.0


Definimos el muestreo de datos completos.

Primero generamos un *y* utilizando *p(y)* y luego seguimos con las probabilidades condicionales.

In [25]:
def sample_from_y(y):
    "Generates a sample of weather conditions based on a specific y"
    o = np.random.choice(p_o_y.index, p=p_o_y[y].values)
    h = np.random.choice(p_h_yo.columns, p=p_h_yo.loc[(y, o)].values)
    w = np.random.choice(p_w_yoh.columns, p=p_w_yoh.loc[(y, o, h)].values)
    t = np.random.choice(p_t_yohw.columns, p=p_t_yohw.loc[(y, o, h, w)].values)
    return [y, o, h, w, t]

def sample():
    "Generates a sample of weather conditions based on a random y"
    y = np.random.choice(p_y.index, p=p_y.values)
    return sample_from_y(y)

In [26]:
samples = np.array([sample() for _ in range(len(df))])
new_df = pd.DataFrame(samples, columns=['Tennis', 'Outlook', 'Humidity', 'Wind', 'Temp'])
new_df

,Tennis,Outlook,Humidity,Wind,Temp
0,Yes,Rain,Normal,Weak,Mild
1,No,Rain,Normal,Strong,Cool
2,Yes,Rain,High,Weak,Mild
3,Yes,Overcast,Normal,Weak,Hot
4,No,Sunny,High,Weak,Mild
5,Yes,Overcast,Normal,Strong,Cool
6,No,Sunny,High,Weak,Hot
7,Yes,Rain,High,Weak,Mild
8,Yes,Overcast,Normal,Strong,Cool
9,Yes,Sunny,Normal,Strong,Mild
